# PointTransformer V2 — Segmentation + Volume Regression Pipeline

A single, beginner-friendly notebook that does two things with **two completely
separate models**:

1. **Segmentation** — a PointTransformerV2 model labels every point as
   `0 = environment` or `1 = wood powder (target)`.
2. **Volume regression** — a *second, independent* PointTransformerV2 model looks
   at only the target points and predicts their **volume** (calibrated against
   `data/gt_volume.csv`).

### How the code is organised (important for readers)
The notebook keeps three kinds of code strictly separate, so each part is easy to
read on its own:

| Section | Contains ONLY |
|---|---|
| **Dataset classes** | how data is loaded, cropped, turned into features |
| **Model classes** | the neural-network architecture (`__init__` + `forward`) |
| **Functions** | training, testing, inference logic |

Nothing is mixed: models don't load data, datasets don't train, training code
lives in plain functions. Run the notebook top to bottom.

### What happens in each phase
- **Training / validation**: both models are trained. Nothing is saved except the
  best model checkpoints.
- **Testing** (held-out files *with* labels): each file is segmented, the target
  part is **saved individually** and **shown in an Open3D window**.
- **Inference** (`data/test`, no labels): same as testing, **plus** the predicted
  **volume** of the target is shown in the window title and printed.

This notebook is meant to run **locally** (it uses Open3D windows, which do not
open on Colab).


## 1 · Configuration & Imports

In [94]:
# ============================================================ IMPORTS
# Everything this notebook needs. Install once if missing:
#   pip install torch laspy open3d numpy scikit-learn matplotlib tqdm
#   pip install torch-scatter torch-cluster -f https://data.pyg.org/whl/torch-<VER>+<CUDA>.html
import os
import glob
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import open3d as o3d
import laspy
from tqdm.auto import tqdm
import pandas as pd

# torch_cluster / torch_scatter provide the fast neighbour + grouping operations
# that PointTransformerV2 attention needs.
from torch_cluster import knn as tc_knn
from torch_scatter import scatter_softmax, scatter_add, scatter_max, scatter_mean

# Pick GPU if available, otherwise CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version :", torch.__version__)
print("Running on      :", DEVICE)

PyTorch version : 2.5.1+cu121
Running on      : cuda


In [95]:
# ============================================================ CONFIG
# Every setting for the whole notebook lives here. Beginners: change values here,
# never inside the code below.
CONFIG = {
    # ---- data folders ----
    "train_dir": "data/train",          # labelled files: used for train/val/test split
    "test_dir":  "data/test",           # unlabelled files: final inference only
    "gt_volume_csv": "data/gt_volume.csv",  # columns: filename,volume (target class only)

    # ---- where results are written ----
    "checkpoint_dir": "checkpoints",    # best models are saved here
    "segment_out_dir": "saved_segments",  # individual segmented target parts (testing/inference)
    "save_format": "ply",               # "ply" (colored, opens in Open3D) or "las"

    # ---- how many points / how the data is split ----
    "num_points": 16384,                 # points per training chunk
    "chunks_per_cloud": 4,              # how many random chunks to draw from each cloud
    "val_ratio": 0.15,
    "test_ratio": 0.15,

    # ---- segmentation model training ----
    "seg_epochs": 200,
    "seg_batch_size": 8,
    "seg_lr": 1e-3,
    "seg_patience": 30,                 # early-stop if val mIoU doesn't improve

    # ---- regression model training ----
    "reg_epochs": 60,
    "reg_batch_size": 8,
    "reg_points": 2048,                 # points sampled from the target object
    "reg_lr": 1e-3,
    "reg_patience": 20,

    # ---- shared model settings ----
    "num_classes": 2,                   # 0 = environment, 1 = target (wood powder)
    "target_class": 1,                  # which class is the object we measure
    "k_neighbors": 16,                  # neighbourhood size for attention
    "normal_k": 16,                     # neighbourhood size for normal estimation

    # ---- resume policy: if a checkpoint already exists, skip that training ----
    "resume": True,

    # ---- visualization ----
    "show_windows": True,               # open Open3D windows during test/inference
    "max_visualize": 50,                 # cap how many files pop up a window

    "seed": 42,
}

# Make output folders and set the checkpoint paths.
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["segment_out_dir"], exist_ok=True)
SEG_CKPT = os.path.join(CONFIG["checkpoint_dir"], "ptv2_segmentation.pth")
REG_CKPT = os.path.join(CONFIG["checkpoint_dir"], "ptv2_regression.pth")

# Make results repeatable.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(CONFIG["seed"])

## 2 · Helper functions for loading point clouds

Small, reusable helpers used by the dataset classes below. Kept separate so the
dataset classes stay short and readable.

In [96]:
# ============================================================ LOAD HELPERS
def load_points_and_labels(path):
    """Read a point-cloud file and return (points[N,3], labels[N] or None).
    Supports .las/.laz (via laspy) and .xyz/.pts/.txt (plain text)."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".las", ".laz"):
        las = laspy.read(path)
        pts = np.column_stack((np.asarray(las.x), np.asarray(las.y),
                               np.asarray(las.z))).astype(np.float64)
        labels = None
        if "classification" in las.point_format.dimension_names:
            labels = np.asarray(las.classification, dtype=np.int64)
        return pts, labels
    # plain text formats: try a few common layouts
    for kwargs in (dict(), dict(delimiter=","), dict(skiprows=1),
                   dict(delimiter=",", skiprows=1)):
        try:
            arr = np.loadtxt(path, **kwargs)
            break
        except ValueError:
            arr = None
    if arr is None:
        arr = np.genfromtxt(path, delimiter=",", skip_header=1)
    arr = np.asarray(arr, dtype=np.float64)
    labels = arr[:, 3].astype(np.int64) if arr.shape[1] >= 4 else None
    return arr[:, :3], labels


def estimate_normals(points, k):
    """Estimate a surface normal for each point using Open3D."""
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(k))
    pcd.orient_normals_to_align_with_direction([0.0, 0.0, 1.0])
    return np.asarray(pcd.normals, dtype=np.float32)


def make_features(points, normal_k):
    """Turn raw XYZ into the 7 input channels the models use:
    [normalized x, y, z, height-above-floor, normal x, y, z].
    These extra channels make the target much easier to separate."""
    center = points.mean(axis=0, keepdims=True)
    scale = max(np.linalg.norm(points - center, axis=1).max(), 1e-9)
    norm_xyz = ((points - center) / scale).astype(np.float32)

    z = points[:, 2]
    height = ((z - z.min()) / max(z.max() - z.min(), 1e-6)).astype(np.float32)

    normals = estimate_normals(points, normal_k)
    return np.column_stack([norm_xyz, height[:, None], normals]).astype(np.float32)


def list_files(folder):
    """Return a sorted list of all supported point-cloud files in a folder."""
    files = []
    for ext in (".las", ".laz", ".xyz", ".pts", ".txt"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)


def load_gt_volumes(csv_path):
    """Read ground-truth volumes into a dict: {filename_without_extension: volume}."""
    volumes = {}
    if not os.path.exists(csv_path):
        print("WARNING: gt_volume.csv not found -> regression cannot be trained.")
        return volumes

    df = pd.read_csv(csv_path)                    # pandas handles header + delimiter

    # find the filename column and the volume column by their content
    # (works no matter what the columns are named or which order they're in)
    filename_col, volume_col = None, None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            volume_col = col
        else:
            filename_col = col
    if filename_col is None:
        filename_col = df.columns[0]
    if volume_col is None:
        volume_col = df.columns[1]

    for _, row in df.iterrows():
        name = os.path.splitext(os.path.basename(str(row[filename_col])))[0]
        volumes[name] = float(row[volume_col])
    return volumes

## 3 · Dataset classes (data only — no training logic here)

Two dataset classes, each doing **one job**:
- `SegmentationDataset` → gives the segmentation model a chunk of points + per-point labels.
- `RegressionDataset` → gives the regression model the target object's points + its true volume.

Both pre-compute the 7-channel features once per file and keep them in memory.

In [97]:
# ============================================================ FILE SPLIT
# Split the labelled training files into train / validation / test groups.
all_train_files = list_files(CONFIG["train_dir"])
assert all_train_files, f"No files found in {CONFIG['train_dir']}"

rng = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_train_files))
n_val = max(1, int(len(all_train_files) * CONFIG["val_ratio"]))
n_test = max(1, int(len(all_train_files) * CONFIG["test_ratio"]))

VAL_FILES   = [all_train_files[i] for i in order[:n_val]]
TEST_FILES  = [all_train_files[i] for i in order[n_val:n_val + n_test]]
TRAIN_FILES = [all_train_files[i] for i in order[n_val + n_test:]]
INFER_FILES = list_files(CONFIG["test_dir"])   # unlabelled, for final inference

print(f"train files      : {len(TRAIN_FILES)}")
print(f"validation files : {len(VAL_FILES)}")
print(f"held-out test    : {len(TEST_FILES)}")
print(f"inference files  : {len(INFER_FILES)}")

# GT_VOLUMES = load_gt_volumes(CONFIG["gt_volume_csv"])
GT_VOLUMES = load_gt_volumes("data/gt_volume.csv")
print(f"ground-truth volumes loaded: {len(GT_VOLUMES)}")

train files      : 33
validation files : 6
held-out test    : 6
inference files  : 18
ground-truth volumes loaded: 63


In [98]:
# ============================================================ FEATURE CACHE
# Computing normals is slow, so we do it ONCE per file and remember the result.
# (Simple dict cache: filename -> {features, labels}.)
FEATURE_CACHE = {}

def get_file_data(path):
    """Return {'points','features','labels'} for a file, computing once and caching."""
    if path in FEATURE_CACHE:
        return FEATURE_CACHE[path]
    points, labels = load_points_and_labels(path)
    if labels is None:
        labels = np.zeros(len(points), dtype=np.int64)
    labels = np.clip(labels, 0, CONFIG["num_classes"] - 1).astype(np.int64)
    features = make_features(points, CONFIG["normal_k"])
    data = {"points": points, "features": features, "labels": labels}
    FEATURE_CACHE[path] = data
    return data

In [99]:
# ============================================================ DATASET: SEGMENTATION
class SegmentationDataset(Dataset):
    """Gives the segmentation model training samples.
    Each sample = one random chunk of `num_points` points from one cloud:
        x -> features shaped (7, num_points)
        y -> per-point labels shaped (num_points,)
    This class ONLY prepares data. It does no training."""

    def __init__(self, files, augment=True):
        self.files = list(files)
        self.augment = augment
        self.chunks_per_cloud = CONFIG["chunks_per_cloud"]
        self.num_points = CONFIG["num_points"]

    def __len__(self):
        # several chunks per cloud
        return len(self.files) * self.chunks_per_cloud

    def __getitem__(self, index):
        file_path = self.files[index // self.chunks_per_cloud]
        data = get_file_data(file_path)
        features = data["features"]
        labels = data["labels"]
        n = len(features)

        # pick `num_points` points: a local sphere crop keeps nearby structure together
        if n <= self.num_points:
            idx = np.random.choice(n, self.num_points, replace=True)
        else:
            seed_point = np.random.randint(n)
            dist2 = ((features[:, :3] - features[seed_point, :3]) ** 2).sum(1)
            idx = np.argpartition(dist2, self.num_points - 1)[:self.num_points]

        x = features[idx].copy()
        y = labels[idx].copy()

        # simple data augmentation: random rotation around the vertical axis
        if self.augment:
            angle = np.random.uniform(0, 2 * np.pi)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            rot = np.array([[cos_a, -sin_a, 0],
                            [sin_a,  cos_a, 0],
                            [0,      0,     1]], dtype=np.float32)
            x[:, 0:3] = x[:, 0:3] @ rot.T          # rotate coordinates
            x[:, 4:7] = x[:, 4:7] @ rot.T          # rotate normals too

        # torch wants shape (channels, points)
        return torch.from_numpy(x.T), torch.from_numpy(y)

In [100]:
# ============================================================ DATASET: REGRESSION
class RegressionDataset(Dataset):
    """Gives the regression model training samples.
    Each sample = the target object's points from one file + that file's true volume:
        x -> features shaped (7, reg_points)
        y -> a single number: the volume
    Only files that (a) contain target points and (b) have a ground-truth volume
    are used. This class ONLY prepares data."""

    def __init__(self, files):
        self.num_points = CONFIG["reg_points"]
        self.samples = []                     # list of (file_path, volume)
        for path in files:
            name = os.path.splitext(os.path.basename(path))[0]
            if name not in GT_VOLUMES:
                continue
            data = get_file_data(path)
            has_target = np.any(data["labels"] == CONFIG["target_class"])
            if has_target:
                self.samples.append((path, GT_VOLUMES[name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        file_path, volume = self.samples[index]
        data = get_file_data(file_path)
        # keep only the target-class points (the object we measure)
        mask = data["labels"] == CONFIG["target_class"]
        target_features = data["features"][mask]
        if len(target_features) < 8:              # safety for tiny objects
            target_features = data["features"]

        # sample a fixed number of points so every batch item has the same size
        replace = len(target_features) < self.num_points
        idx = np.random.choice(len(target_features), self.num_points, replace=replace)
        x = target_features[idx].copy()

        # predict log(1+volume): keeps training stable across small and large piles
        y = np.log1p(volume).astype(np.float32)
        return torch.from_numpy(x.T), torch.tensor(y, dtype=torch.float32)

## 4 · Model classes (architecture only — no training logic here)

Two **completely separate** PointTransformerV2 models:
- `PTv2Segmentation` → outputs a class score for every point.
- `PTv2Regression` → outputs one number (the volume) for the whole object.

They share the same building blocks (grouped vector attention + grid pooling)
but are independent networks — neither reuses the other's weights.

The shared building blocks are defined first, then each model uses them.

In [101]:
# ============================================================ SHARED BUILDING BLOCKS
IN_CHANNELS = 7   # our features: xyz(3) + height(1) + normal(3)

class GroupedVectorAttention(nn.Module):
    """The core PointTransformerV2 attention: each point attends to its k nearest
    neighbours, using the relative position between points as extra information."""
    def __init__(self, channels, groups=6, k=16):
        super().__init__()
        assert channels % groups == 0
        self.k = k
        self.groups = groups
        self.group_channels = channels // groups
        self.to_query = nn.Linear(channels, channels)
        self.to_key   = nn.Linear(channels, channels)
        self.to_value = nn.Linear(channels, channels)
        # two small networks that turn relative position into weights/biases
        self.pos_multiplier = nn.Sequential(nn.Linear(3, channels), nn.ReLU(),
                                            nn.Linear(channels, channels))
        self.pos_bias = nn.Sequential(nn.Linear(3, channels), nn.ReLU(),
                                      nn.Linear(channels, channels))
        self.weight_net = nn.Sequential(nn.Linear(channels, channels), nn.ReLU(),
                                        nn.Linear(channels, groups))

    def forward(self, x, pos, batch):
        # find k nearest neighbours for every point (within the same cloud)
        edges = tc_knn(pos, pos, self.k, batch, batch)
        center, neighbour = edges[0], edges[1]
        rel_pos = pos[center] - pos[neighbour]

        bias = self.pos_bias(rel_pos)
        relation = (self.to_query(x)[center] - self.to_key(x)[neighbour]) \
            * self.pos_multiplier(rel_pos) + bias
        weights = scatter_softmax(self.weight_net(relation), center, dim=0)

        values = (self.to_value(x)[neighbour] + bias).view(-1, self.groups,
                                                           self.group_channels)
        out = scatter_add(values * weights.unsqueeze(-1), center, dim=0,
                          dim_size=x.size(0))
        return out.view(-1, self.groups * self.group_channels)


class AttentionBlock(nn.Module):
    """One attention layer + one small MLP, each with a residual connection."""
    def __init__(self, channels, groups=6, k=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.attention = GroupedVectorAttention(channels, groups, k)
        self.mlp = nn.Sequential(nn.Linear(channels, channels * 2), nn.ReLU(),
                                 nn.Linear(channels * 2, channels))

    def forward(self, x, pos, batch):
        x = x + self.attention(self.norm1(x), pos, batch)
        x = x + self.mlp(self.norm2(x))
        return x


class GridPooling(nn.Module):
    """Shrink the cloud by merging all points that fall in the same voxel cell.
    Returns the pooled features plus a map so we can un-pool later."""
    def __init__(self, in_ch, out_ch, grid_size):
        super().__init__()
        self.grid_size = grid_size
        self.project = nn.Sequential(nn.Linear(in_ch, out_ch), nn.ReLU())

    def forward(self, x, pos, batch):
        voxel = torch.floor(pos / self.grid_size).long()
        key = torch.cat([batch.unsqueeze(1), voxel], dim=1)
        _, cluster = torch.unique(key, dim=0, return_inverse=True)
        pooled_x, _ = scatter_max(self.project(x), cluster, dim=0)
        pooled_pos = scatter_mean(pos, cluster, dim=0)
        pooled_batch = scatter_max(batch, cluster, dim=0)[0]
        return pooled_x, pooled_pos, pooled_batch, cluster

In [102]:
# ============================================================ MODEL 1: SEGMENTATION
class PTv2Segmentation(nn.Module):
    """PointTransformerV2 for per-point segmentation.
    Input : features (batch, 7, num_points)
    Output: class scores (batch, num_classes, num_points)
    Encoder shrinks the cloud twice, decoder grows it back to every point."""

    def __init__(self, num_classes, k=16, dims=(48, 96, 192), groups=6):
        super().__init__()
        d0, d1, d2 = dims
        self.embed = nn.Sequential(nn.Linear(IN_CHANNELS, d0), nn.ReLU(),
                                   nn.Linear(d0, d0))
        # encoder
        self.enc1 = AttentionBlock(d0, groups, k)
        self.pool1 = GridPooling(d0, d1, grid_size=0.08)
        self.enc2 = AttentionBlock(d1, groups, k)
        self.pool2 = GridPooling(d1, d2, grid_size=0.16)
        self.enc3 = AttentionBlock(d2, groups, k)
        # decoder (un-pool by copying pooled features back to member points)
        self.up2 = nn.Sequential(nn.Linear(d2 + d1, d1), nn.ReLU())
        self.dec2 = AttentionBlock(d1, groups, k)
        self.up1 = nn.Sequential(nn.Linear(d1 + d0, d0), nn.ReLU())
        self.dec1 = AttentionBlock(d0, groups, k)
        # final per-point classifier
        self.head = nn.Sequential(nn.LayerNorm(d0), nn.Linear(d0, 128), nn.ReLU(),
                                  nn.Dropout(0.4), nn.Linear(128, num_classes))

    def forward(self, x):
        batch_size, channels, num_points = x.shape
        # flatten (B, C, N) -> (B*N, C); remember which cloud each point belongs to
        flat = x.permute(0, 2, 1).reshape(-1, channels).contiguous()
        pos = flat[:, :3].contiguous()                 # first 3 channels are xyz
        batch = torch.arange(batch_size, device=x.device).repeat_interleave(num_points)

        h0 = self.enc1(self.embed(flat), pos, batch)
        h1, pos1, batch1, cluster1 = self.pool1(h0, pos, batch)
        h1 = self.enc2(h1, pos1, batch1)
        h2, pos2, batch2, cluster2 = self.pool2(h1, pos1, batch1)
        h2 = self.enc3(h2, pos2, batch2)

        up1 = self.dec2(self.up2(torch.cat([h1, h2[cluster2]], dim=1)), pos1, batch1)
        up0 = self.dec1(self.up1(torch.cat([h0, up1[cluster1]], dim=1)), pos, batch)

        logits = self.head(up0)                        # (B*N, num_classes)
        return logits.view(batch_size, num_points, -1).permute(0, 2, 1)

In [103]:
# ============================================================ MODEL 2: REGRESSION
class PTv2Regression(nn.Module):
    """PointTransformerV2 for volume regression — a SEPARATE model.
    Input : features of the target object (batch, 7, reg_points)
    Output: one number per object (batch,) = predicted log(1+volume)
    Same attention encoder, but instead of labelling points it pools everything
    into one vector and predicts a single value."""

    def __init__(self, k=16, dims=(48, 96, 192), groups=6):
        super().__init__()
        d0, d1, d2 = dims
        self.embed = nn.Sequential(nn.Linear(IN_CHANNELS, d0), nn.ReLU(),
                                   nn.Linear(d0, d0))
        self.enc1 = AttentionBlock(d0, groups, k)
        self.pool1 = GridPooling(d0, d1, grid_size=0.08)
        self.enc2 = AttentionBlock(d1, groups, k)
        self.pool2 = GridPooling(d1, d2, grid_size=0.16)
        self.enc3 = AttentionBlock(d2, groups, k)
        # from a single pooled vector -> one number
        self.head = nn.Sequential(nn.Linear(d2 * 2, 128), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, x):
        batch_size, channels, num_points = x.shape
        flat = x.permute(0, 2, 1).reshape(-1, channels).contiguous()
        pos = flat[:, :3].contiguous()
        batch = torch.arange(batch_size, device=x.device).repeat_interleave(num_points)

        h0 = self.enc1(self.embed(flat), pos, batch)
        h1, pos1, batch1, _ = self.pool1(h0, pos, batch)
        h1 = self.enc2(h1, pos1, batch1)
        h2, pos2, batch2, _ = self.pool2(h1, pos1, batch1)
        h2 = self.enc3(h2, pos2, batch2)

        # summarise the whole object with max + mean pooling, then predict
        pooled_max, _ = scatter_max(h2, batch2, dim=0, dim_size=batch_size)
        pooled_mean = scatter_mean(h2, batch2, dim=0, dim_size=batch_size)
        summary = torch.cat([pooled_max, pooled_mean], dim=1)
        return self.head(summary).squeeze(-1)          # (batch,)

## 5 · Metric helpers (plain functions)

In [104]:
# ============================================================ METRICS
def compute_confusion(true_labels, pred_labels, num_classes):
    """Count how often each true class was predicted as each class."""
    k = num_classes
    return np.bincount(true_labels * k + pred_labels,
                       minlength=k * k).reshape(k, k)

def confusion_to_scores(conf):
    """Turn a confusion matrix into accuracy, per-class IoU, and mean IoU."""
    conf = conf.astype(np.float64)
    accuracy = np.diag(conf).sum() / max(conf.sum(), 1)
    ious = []
    for c in range(conf.shape[0]):
        tp = conf[c, c]
        fp = conf[:, c].sum() - tp
        fn = conf[c, :].sum() - tp
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float("nan")
        ious.append(iou)
    mean_iou = float(np.nanmean(ious))
    return {"accuracy": float(accuracy), "iou_per_class": ious, "miou": mean_iou}

## 6 · Training functions (plain functions — models/datasets stay untouched)

`train_segmentation()` and `train_regression()` each: build the right dataset,
build the right model, loop over epochs, keep the best weights, and save a
checkpoint. If `CONFIG["resume"]` is on and a checkpoint already exists, they
just load it and skip training (same policy as before).

In [105]:
# ============================================================ TRAIN: SEGMENTATION
def train_segmentation():
    """Train PTv2Segmentation. Returns the trained model."""
    model = PTv2Segmentation(CONFIG["num_classes"], k=CONFIG["k_neighbors"]).to(DEVICE)

    # resume: skip training if we already have a saved model
    if CONFIG["resume"] and os.path.exists(SEG_CKPT):
        print(f"[resume] found {SEG_CKPT} -> loading, skipping segmentation training")
        model.load_state_dict(torch.load(SEG_CKPT, map_location=DEVICE)["model_state"])
        return model

    train_loader = DataLoader(SegmentationDataset(TRAIN_FILES, augment=True),
                              batch_size=CONFIG["seg_batch_size"], shuffle=True,
                              drop_last=True)
    val_loader = DataLoader(SegmentationDataset(VAL_FILES, augment=False),
                            batch_size=CONFIG["seg_batch_size"], shuffle=False)

    # class weights help when one class has far more points than the other
    counts = np.zeros(CONFIG["num_classes"])
    for f in TRAIN_FILES[:30]:
        counts += np.bincount(get_file_data(f)["labels"],
                              minlength=CONFIG["num_classes"])
    weights = 1.0 / np.clip(counts / counts.sum(), 1e-6, None)
    weights = torch.tensor(weights / weights.sum() * CONFIG["num_classes"],
                           dtype=torch.float32, device=DEVICE)

    loss_fn = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["seg_lr"],
                                  weight_decay=1e-4)

    best_miou = -1.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, CONFIG["seg_epochs"] + 1):
        # ---- train one epoch ----
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

        # ---- validate ----
        model.eval()
        conf = np.zeros((CONFIG["num_classes"],) * 2, dtype=np.int64)
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x).argmax(dim=1)
                conf += compute_confusion(y.cpu().numpy().ravel(),
                                          preds.cpu().numpy().ravel(),
                                          CONFIG["num_classes"])
        scores = confusion_to_scores(conf)
        note = ""
        if scores["miou"] > best_miou + 1e-4:
            best_miou = scores["miou"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            note = "  <- best so far"
        else:
            epochs_without_improvement += 1
        print(f"epoch {epoch:3d} | val accuracy {scores['accuracy']:.4f} | "
              f"val mIoU {scores['miou']:.4f}{note}")

        if epochs_without_improvement >= CONFIG["seg_patience"]:
            print(f"early stop: no improvement for {CONFIG['seg_patience']} epochs")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save({"model_state": model.state_dict(), "best_miou": best_miou}, SEG_CKPT)
    print(f"saved best segmentation model -> {SEG_CKPT} (val mIoU {best_miou:.4f})")
    return model

In [106]:
# ============================================================ TRAIN: REGRESSION
def train_regression():
    """Train PTv2Regression on target-object points vs ground-truth volume.
    Returns the trained model (or None if there is not enough GT data)."""
    model = PTv2Regression(k=CONFIG["k_neighbors"]).to(DEVICE)

    if CONFIG["resume"] and os.path.exists(REG_CKPT):
        print(f"[resume] found {REG_CKPT} -> loading, skipping regression training")
        model.load_state_dict(torch.load(REG_CKPT, map_location=DEVICE)["model_state"])
        return model

    full_dataset = RegressionDataset(TRAIN_FILES + VAL_FILES)
    if len(full_dataset) < 4:
        print("Not enough files with both target points and GT volume -> "
              "skipping regression training.")
        return None

    # simple train/val split inside the regression samples
    n_val = max(1, int(0.2 * len(full_dataset)))
    val_subset = torch.utils.data.Subset(full_dataset, list(range(n_val)))
    train_subset = torch.utils.data.Subset(full_dataset, list(range(n_val, len(full_dataset))))

    train_loader = DataLoader(train_subset, batch_size=CONFIG["reg_batch_size"],
                              shuffle=True, drop_last=len(train_subset) > CONFIG["reg_batch_size"])
    val_loader = DataLoader(val_subset, batch_size=CONFIG["reg_batch_size"])

    loss_fn = nn.SmoothL1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["reg_lr"],
                                  weight_decay=1e-4)

    best_mae = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, CONFIG["reg_epochs"] + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()

        # validate: compare predicted volume with true volume (undo the log1p)
        model.eval()
        errors = []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE)
                pred_volume = np.expm1(model(x).cpu().numpy())
                true_volume = np.expm1(y.numpy())
                errors.extend(np.abs(pred_volume - true_volume).tolist())
        mae = float(np.mean(errors)) if errors else float("inf")
        note = ""
        if mae < best_mae - 1e-6:
            best_mae = mae
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            note = "  <- best so far"
        else:
            epochs_without_improvement += 1
        print(f"epoch {epoch:3d} | val volume MAE {mae:.4f}{note}")

        if epochs_without_improvement >= CONFIG["reg_patience"]:
            print(f"early stop: no improvement for {CONFIG['reg_patience']} epochs")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save({"model_state": model.state_dict(), "best_mae": best_mae}, REG_CKPT)
    print(f"saved best regression model -> {REG_CKPT} (val MAE {best_mae:.4f})")
    return model

## 7 · Prediction helpers (segment a whole cloud; estimate a volume)

In [107]:
# ============================================================ PREDICT HELPERS
@torch.no_grad()
def segment_whole_cloud(seg_model, features):
    """Label every point in a cloud. We feed the cloud in fixed-size chunks
    (the model expects a fixed number of points) and stitch the answers back."""
    seg_model.eval()
    num_points = CONFIG["num_points"]
    batch_size = CONFIG["seg_batch_size"]
    n = len(features)

    order = np.random.RandomState(0).permutation(n)
    pad = (num_points - n % num_points) % num_points
    if pad:
        order = np.concatenate([order, order[:pad]])
    chunks = order.reshape(-1, num_points)

    predictions = np.zeros(n, dtype=np.int64)
    for start in range(0, len(chunks), batch_size):
        chunk_ids = chunks[start:start + batch_size]
        x = torch.from_numpy(features[chunk_ids].transpose(0, 2, 1)).to(DEVICE)
        preds = seg_model(x).argmax(dim=1).cpu().numpy()
        predictions[chunk_ids.ravel()] = preds.ravel()
    return predictions


@torch.no_grad()
def estimate_volume(reg_model, target_features):
    """Predict the volume of one object from its target-class features."""
    if reg_model is None or len(target_features) < 8:
        return float("nan")
    reg_model.eval()
    num_points = CONFIG["reg_points"]
    replace = len(target_features) < num_points
    idx = np.random.RandomState(0).choice(len(target_features), num_points, replace=replace)
    x = torch.from_numpy(target_features[idx].T).unsqueeze(0).to(DEVICE)
    pred_log = reg_model(x).item()
    return max(float(np.expm1(pred_log)), 0.0)


def save_segment(points, filename):
    """Save the target-object points as a colored .ply (or .las) file."""
    out_dir = CONFIG["segment_out_dir"]
    if CONFIG["save_format"] == "las":
        header = laspy.LasHeader(point_format=3, version="1.2")
        las = laspy.LasData(header)
        las.x, las.y, las.z = points[:, 0], points[:, 1], points[:, 2]
        path = os.path.join(out_dir, filename + "_target.las")
        las.write(path)
    else:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
        pcd.paint_uniform_color([0.0, 0.8, 0.0])       # green target
        path = os.path.join(out_dir, filename + "_target.ply")
        o3d.io.write_point_cloud(path, pcd)
    return path


# def show_segment_window(points_all, predictions, title):
#     """Open an Open3D window: target points green, everything else gray."""
#     if not CONFIG["show_windows"]:
#         return
#     colors = np.zeros((len(points_all), 3))
#     is_target = predictions == CONFIG["target_class"]
#     colors[is_target] = [0.0, 0.8, 0.0]                # green = target
#     colors[~is_target] = [0.35, 0.35, 0.35]            # gray = others
#     pcd = o3d.geometry.PointCloud()
#     pcd.points = o3d.utility.Vector3dVector(points_all.astype(np.float64)-np.mean(points_all.astype(np.float64),axis=0))
#     pcd.colors = o3d.utility.Vector3dVector(colors)
#     o3d.visualization.draw_geometries([pcd], window_name=title, width=1280, height=800)


def show_segment_window(points_all, predictions, title):
    """Open an Open3D window: showing ONLY the segmented target points."""
    if not CONFIG["show_windows"]:
        return
    
    # 1. Sudhumatro target class-er point gulo filter korbo
    is_target = predictions == CONFIG["target_class"]
    target_points = points_all[is_target]
    
    # Jodi kono target point na thake, tahole window open korbe na
    if len(target_points) == 0:
        return

    # 2. Shob target point-ke shobuj (green) rong korbo
    colors = np.zeros((len(target_points), 3))
    colors[:] = [0.0, 0.8, 0.0]                # green = target
    
    # 3. Open3D PointCloud toiri kora ebong center kora jate thikmoto dekha jay
    pcd = o3d.geometry.PointCloud()
    centered_points = target_points.astype(np.float64) - np.mean(target_points.astype(np.float64), axis=0)
    pcd.points = o3d.utility.Vector3dVector(centered_points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    # 4. Window show kora
    o3d.visualization.draw_geometries([pcd], window_name=title, width=1280, height=800)

## 8 · Testing function (held-out labelled files)

For each held-out test file: segment it, **save the target part individually**,
and **show it in an Open3D window**. (No volume here — this phase is about
checking segmentation quality.) It also prints the segmentation score since these
files have labels.

In [108]:
# ============================================================ TEST
def test(seg_model):
    """Run segmentation on the held-out test files: score, save, and visualize."""
    print("\n===== TESTING (held-out labelled files) =====")
    total_conf = np.zeros((CONFIG["num_classes"],) * 2, dtype=np.int64)
    shown = 0

    for path in TEST_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        data = get_file_data(path)
        predictions = segment_whole_cloud(seg_model, data["features"])

        # score (these files are labelled)
        total_conf += compute_confusion(data["labels"], predictions,
                                        CONFIG["num_classes"])

        # save the target part on its own
        target_points = data["points"][predictions == CONFIG["target_class"]]
        if len(target_points) > 0:
            saved = save_segment(target_points, name)
            print(f"{name}: {len(target_points)} target points -> saved {saved}")
        else:
            print(f"{name}: no target points predicted")

        # visualize (cap how many windows open)
        if shown < CONFIG["max_visualize"]:
            show_segment_window(data["points"], predictions,
                                title=f"TEST | {name} | green = target")
            shown += 1

    scores = confusion_to_scores(total_conf)
    print(f"\nOverall test accuracy : {scores['accuracy']:.4f}")
    print(f"Overall test mIoU     : {scores['miou']:.4f}")
    for c in range(CONFIG["num_classes"]):
        print(f"  class {c} IoU: {scores['iou_per_class'][c]:.4f}")

## 9 · Inference function (`data/test`, unlabelled)

Same as testing — segment, save target part, visualize — **plus** it uses the
regression model to estimate the target's **volume** and shows it in the window
title and console.

In [109]:
# ============================================================ INFERENCE
def inference(seg_model, reg_model):
    """Segment unlabelled files, save + visualize the target, and show its volume."""
    print("\n===== INFERENCE (data/test, unlabelled) =====")
    if not INFER_FILES:
        print("No files in data/test -> nothing to do.")
        return

    shown = 0
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        data = get_file_data(path)
        predictions = segment_whole_cloud(seg_model, data["features"])

        # pull out the target object
        target_mask = predictions == CONFIG["target_class"]
        target_points = data["points"][target_mask]
        target_features = data["features"][target_mask]

        # estimate its volume with the regression model
        volume = estimate_volume(reg_model, target_features)

        # save the target part
        if len(target_points) > 0:
            saved = save_segment(target_points, name)
            print(f"{name}: {len(target_points)} target points | "
                  f"estimated volume = {volume:.4f} -> saved {saved}")
        else:
            print(f"{name}: no target points predicted | volume = n/a")

        # visualize with the volume in the window title
        if shown < CONFIG["max_visualize"]:
            show_segment_window(data["points"], predictions,
                                title=f"INFERENCE | {name} | volume = {volume:.4f}")
            shown += 1

## 10 · Run everything

Train (or resume) both models, then test, then run inference.

In [110]:
# ============================================================ RUN
# 1) segmentation model
seg_model = train_segmentation()

# 2) regression model (separate, independent model)
reg_model = train_regression()

# 3) testing: save + visualize target parts of the held-out labelled files
test(seg_model)

# 4) inference: same, plus predicted volume, on the unlabelled data/test files
inference(seg_model, reg_model)

print("\nAll done.")

epoch   1 | val accuracy 0.6004 | val mIoU 0.3002  <- best so far
epoch   2 | val accuracy 0.4616 | val mIoU 0.2308
epoch   3 | val accuracy 0.3908 | val mIoU 0.1954
epoch   4 | val accuracy 0.6134 | val mIoU 0.4350  <- best so far
epoch   5 | val accuracy 0.7491 | val mIoU 0.5907  <- best so far
epoch   6 | val accuracy 0.8529 | val mIoU 0.7307  <- best so far
epoch   7 | val accuracy 0.7483 | val mIoU 0.5688
epoch   8 | val accuracy 0.7426 | val mIoU 0.5634
epoch   9 | val accuracy 0.7376 | val mIoU 0.5620
epoch  10 | val accuracy 0.7196 | val mIoU 0.5432
epoch  11 | val accuracy 0.7322 | val mIoU 0.5774
epoch  12 | val accuracy 0.7366 | val mIoU 0.5613
epoch  13 | val accuracy 0.8387 | val mIoU 0.7009
epoch  14 | val accuracy 0.7432 | val mIoU 0.5662
epoch  15 | val accuracy 0.7977 | val mIoU 0.6227
epoch  16 | val accuracy 0.7232 | val mIoU 0.5323
epoch  17 | val accuracy 0.7523 | val mIoU 0.5544
epoch  18 | val accuracy 0.7901 | val mIoU 0.5296
epoch  19 | val accuracy 0.7795 | va

KeyboardInterrupt: 